# TechChallenge3_Bronze_to_Silver

In [1]:
import sys
import re
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# 1. Inicialização
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

print("Iniciando processamento: Camada Bronze -> Silver...")

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Trying to create a Glue session for the kernel.
Session Type: glueetl
Session ID: bceda9c3-cbdf-4007-a568-3ec54e296af2
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
Waiting for session bceda9c3-cbdf-4007-a568-3ec54e296af2 to get into ready status...
Session bceda9c3-cbdf-4007-a568-3ec54e296af2 has been created.
Iniciando processamento: Camada Bronze -> Silver...


In [2]:
# 2. Leitura dos dados brutos
df_2023 = spark.read.option("header", "true").option("inferSchema", "true").csv("s3://tech3-datahackers-datalake/bronze/year=2023/")
df_2024 = spark.read.option("header", "true").option("inferSchema", "true").csv("s3://tech3-datahackers-datalake/bronze/year=2024/")
df_2025 = spark.read.option("header", "true").option("inferSchema", "true").csv("s3://tech3-datahackers-datalake/bronze/year=2025/")

def limpar_cabecalhos_dataframe(df):
    """
    Remove pontos, espaços e caracteres especiais de todas as colunas
    para que o Spark não quebre com Syntax Error.
    """
    novas_colunas = []
    for c in df.columns:
        # Substitui pontos e espaços por underline e remove caracteres especiais
        nome_limpo = c.replace(".", "_").replace(" ", "_").replace("?", "")
        nome_limpo = re.sub(r'[^a-zA-Z0-9_]', '_', nome_limpo)
        nome_limpo = re.sub(r'_+', '_', nome_limpo).strip('_')
        novas_colunas.append(nome_limpo)
    
    # Aplica os novos nomes limpos ao DataFrame
    return df.toDF(*novas_colunas)

# 3. Limpeza estrutural de todos os cabeçalhos
df_2023_clean = limpar_cabecalhos_dataframe(df_2023)
df_2024_clean = limpar_cabecalhos_dataframe(df_2024)
df_2025_clean = limpar_cabecalhos_dataframe(df_2025)

# 4. Função de Padronização (Mapeamento atualizado para os nomes limpos)
def padronizar_colunas(df_limpo, year):
    mapping = {}
    
    if year == 2023:
        mapping = {
            "P1_a_Idade": "age",
            "P1_b_Genero": "gender",
            "P1_c_Cor_raca_etnia": "ethnicity",
            "P1_i_2_Regiao_onde_mora": "region",
            "P2_b_Setor": "industry",
            "P2_f_Cargo_Atual": "current_role",
            "P2_g_Nivel": "seniority_level",
            "P2_h_Faixa_salarial": "salary_range",
            "P2_r_Atualmente_qual_a_sua_forma_de_trabalho": "work_model",
            "P4_h_Dentre_as_op_es_listadas_qual_sua_Cloud_preferida": "preferred_cloud",
            "P3_e_AI_Generativa_uma_prioridade_em_sua_empresa": "ai_priority",
            "P1_l_Nivel_de_Ensino": "education_level",
            "P2_i_Quanto_tempo_de_experi_ncia_na_rea_de_dados_voc_tem": "experience_time",
            "P2_k_Voc_est_satisfeito_na_sua_empresa_atual": "job_satisfaction",
            "P4_f_Entre_as_linguagens_listadas_abaixo_qual_a_sua_preferida": "preferred_language",
            "P4_k_Qual_sua_ferramenta_de_BI_preferida": "preferred_bi"
        }
    elif year == 2024:
        mapping = {
            "1_a_idade": "age",
            "1_b_genero": "gender",
            "1_c_cor_raca_etnia": "ethnicity",
            "1_i_2_regiao_onde_mora": "region",
            "2_b_setor": "industry",
            "2_f_cargo_atual": "current_role",
            "2_g_nivel": "seniority_level",
            "2_h_faixa_salarial": "salary_range",
            "2_r_modelo_de_trabalho_atual": "work_model",
            "4_h_cloud_dia_a_dia": "preferred_cloud",
            "3_e_ai_generativa_e_llm_uma_prioridade": "ai_priority",
            "1_l_nivel_de_ensino": "education_level",
            "2_i_tempo_de_experiencia_em_dados": "experience_time",
            "2_k_satisfeito_atualmente": "job_satisfaction",
            "4_f_linguagem_preferida": "preferred_language",
            "4_k_ferramenta_de_bi_preferida": "preferred_bi"
        }
    elif year == 2025:
        mapping = {
            "1_a_idade": "age",
            "1_b_genero": "gender",
            "1_c_cor_raca_etnia": "ethnicity",
            "1_i_2_regiao_onde_mora": "region",
            "2_b_setor": "industry",
            "2_f_cargo_atual": "current_role",
            "2_g_nivel": "seniority_level",
            "2_h_faixa_salarial": "salary_range",
            "2_q_modelo_de_trabalho_atual": "work_model",
            "4_e_cloud_dia_a_dia": "preferred_cloud",
            "3_e_ai_generativa_e_llm_uma_prioridade": "ai_priority",
            "1_l_nivel_de_ensino": "education_level",
            "2_i_tempo_de_experiencia_em_dados": "experience_time",
            "2_k_satisfeito_atualmente": "job_satisfaction",
            "4_c_linguagem_preferida": "preferred_language",
            "4_h_ferramenta_de_bi_preferida": "preferred_bi"
        }

    colunas_selecionadas = [F.lit(year).cast("string").alias("year")]
    
    # Verifica dinamicamente o nome da coluna limpa
    for orig_col, new_col_name in mapping.items():
        # Busca por aproximação no nome limpo (para ignorar acentos cortados)
        col_encontrada = None
        for c in df_limpo.columns:
            if orig_col in c:
                col_encontrada = c
                break
                
        if col_encontrada:
            # A MÁGICA ACONTECE AQUI: Adicionamos o .cast("string")
            colunas_selecionadas.append(F.col(col_encontrada).cast("string").alias(new_col_name))
        else:
            # E AQUI TAMBÉM: Se for nulo, forçamos que o nulo seja do tipo string
            colunas_selecionadas.append(F.lit(None).cast("string").alias(new_col_name))
            
    return df_limpo.select(colunas_selecionadas)

# 5. Processamento e União
df_23 = padronizar_colunas(df_2023_clean, 2023)
df_24 = padronizar_colunas(df_2024_clean, 2024)
df_25 = padronizar_colunas(df_2025_clean, 2025)

df_silver = df_23.unionByName(df_24).unionByName(df_25)

# 6. Escrita na Silver
output_path = "s3://tech3-datahackers-datalake/silver/datahackers_unified/"
df_silver.write.mode("overwrite").partitionBy("year").parquet(output_path)

print("Processamento Bronze -> Silver concluído com sucesso!")#### Optional: Run this cell to see available notebook commands ("magics").

Processamento Bronze -> Silver concluído com sucesso!
